# 20 · Gray-Box AIA72 AR-Disjoint and Event-Disjoint Audit

Phase D audit for the frozen 72-hour Gray-Box experiment. The notebook verifies region-component separation from training, reports AR-level sensitivity metrics, and—when exact retained event-linkage artifacts are available—reports unique-event detection, misses and lead times.

**No fitting, threshold tuning, policy selection or redesign is performed.** Cycle 24 is development/descriptive, Cycle 25 is diagnostic/post-hoc for v3, and supplementary 2026 is already-spent locked evidence.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

ROOT=Path('/home/abmoses2000')
SEEDS=[17,29,43]
PRIMARY_ALPHA=.10
PRIMARY_PCT=.90

MASTER=ROOT/'graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz'
MASTER_SHA='17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46'
SHARP=ROOT/'sharp72_predictions_frozen_20261002.csv.gz'
SHARP_SHA='7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b'
CASES=ROOT/'graybox_aia72_inference_v1_20261004/inputs/cases.csv.gz'
MOND=ROOT/'graybox_aia72_conformal_mondrian_v2_20261005/mondrian_thresholds.csv'
CUTS=ROOT/'graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv'
ALARM=ROOT/'graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005/alarm_thresholds.csv'

OUT=ROOT/'graybox_aia72_ar_event_disjoint_v1_20261006'
if OUT.exists():
    raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
OUT.mkdir(parents=True)

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for c in iter(lambda:f.read(4*1024*1024),b''):
            h.update(c)
    return h.hexdigest()

def require(c,m):
    if not c:
        raise ValueError(m)

for p in [MASTER,SHARP,CASES,MOND,CUTS,ALARM]:
    require(p.is_file(),f'Missing frozen input: {p}')
require(sha256(MASTER)==MASTER_SHA,'Master SHA changed')
require(sha256(SHARP)==SHARP_SHA,'SHARP SHA changed')

a=pd.read_csv(MASTER,low_memory=False)
s=pd.read_csv(SHARP,low_memory=False)
cases=pd.read_csv(CASES,low_memory=False)

for d in [a,s,cases]:
    d['forecast_case_id']=d['forecast_case_id'].astype(str)
    require(d['forecast_case_id'].is_unique,'Duplicate forecast_case_id')

require('region_component_id' in cases.columns,'Frozen cases lack region_component_id')
require('role' in cases.columns,'Frozen cases lack role')

m=a.merge(s,on='forecast_case_id',how='inner',suffixes=('_aia','_sharp'),validate='one_to_one')
m=m.merge(cases[['forecast_case_id','region_component_id','HARPNUM','NOAA_AR_clean','role','issue_utc']],
          on='forecast_case_id',how='left',suffixes=('','_case'),validate='one_to_one')
require(len(m)==len(a),'Prediction/case alignment loss')
require(m['region_component_id'].notna().all(),'Missing region_component_id in scored rows')

la='label_aia' if 'label_aia' in m.columns else 'label'
ls='label_sharp' if 'label_sharp' in m.columns else la
m['label_final']=pd.to_numeric(m[la],errors='raise').astype(int)
require(np.array_equal(m['label_final'].to_numpy(),pd.to_numeric(m[ls],errors='raise').astype(int).to_numpy()),'Label mismatch')

role_col='role_aia' if 'role_aia' in m.columns else ('role_sharp' if 'role_sharp' in m.columns else 'role')
m['role_final']=m[role_col].astype(str)
m['issue_dt']=pd.to_datetime(m['issue_utc_case'] if 'issue_utc_case' in m.columns else m['issue_utc'],utc=True,errors='coerce')
require(m['issue_dt'].notna().all(),'Unresolved issue time')
m['year']=m['issue_dt'].dt.year.astype(int)

m['sharp_probability']=pd.to_numeric(m['probability_gru_mean'],errors='coerce')
m['aia_probability']=pd.to_numeric(m['probability'],errors='coerce')
m['fusion_probability']=.5*(m['sharp_probability']+m['aia_probability'])

def seed_cols(side):
    out=[]
    for z in SEEDS:
        c=f'probability_seed_{z}_{side}'
        if c in m.columns: out.append(c)
        elif f'probability_seed_{z}' in m.columns: out.append(f'probability_seed_{z}')
        else: raise KeyError(f'Missing seed {z} for {side}')
    return out

AS=m[seed_cols('aia')].astype(float).to_numpy()
SS=m[seed_cols('sharp')].astype(float).to_numpy()
FS=.5*(AS+SS)
m['fusion_seed_std']=np.nanstd(FS,axis=1,ddof=1)
m['sharp_aia_probability_gap']=abs(m['sharp_probability']-m['aia_probability'])
q=np.clip(m['fusion_probability'].to_numpy(float),1e-12,1-1e-12)
m['fusion_entropy']=-(q*np.log(q)+(1-q)*np.log(1-q))

def regime(r):
    if r.role_final=='retrospective_cycle25': return 'cycle25_2021_2025_diagnostic'
    if r.role_final=='supplementary_2026': return 'supplementary_2026_locked'
    if r.year<=2019: return 'cycle24_earlier_development'
    return 'other_or_gap'
m['regime']=m.apply(regime,axis=1)

# ------------------------------------------------------------
# Frozen branch alarm decisions and v3 q90 reconstruction
# ------------------------------------------------------------
alarm=pd.read_csv(ALARM)
ath={r.branch:float(r.threshold) for r in alarm.itertuples()}
for b,pcol in [('sharp','sharp_probability'),('aia','aia_probability'),('sharp_aia','fusion_probability')]:
    m[f'{b}_alarm']=(m[pcol].to_numpy(float)>=ath[b]).astype(int)

mond=pd.read_csv(MOND)
mq={}
for b in ['sharp','aia','sharp_aia']:
    z=mond[(mond.branch==b)&np.isclose(mond.alpha.astype(float),PRIMARY_ALPHA)]
    require(len(z)==1,f'Missing Mondrian threshold: {b}')
    mq[b]=(float(z.iloc[0].qhat_class0),float(z.iloc[0].qhat_class1))

p=m['fusion_probability'].to_numpy(float)
q0,q1=mq['sharp_aia']
inc0=p<=q0; inc1=(1-p)<=q1
fusion_single=(inc0&~inc1)|(~inc0&inc1)

cuts=pd.read_csv(CUTS)
def cutoff(metric):
    z=cuts[(cuts.metric==metric)&np.isclose(cuts.percentile.astype(float),PRIMARY_PCT)]
    require(len(z)==1,f'Missing q90 cutoff: {metric}')
    return float(z.iloc[0].cutoff)

aia_status='input_status' if 'input_status' in m.columns else ('input_status_aia' if 'input_status_aia' in m.columns else None)
aia_ok=np.isfinite(m['aia_probability'])
if aia_status is not None:
    aia_ok &= m[aia_status].astype(str).eq('ok').to_numpy()
sharp_ok=np.isfinite(m['sharp_probability'])
fusion_ok=aia_ok&sharp_ok&np.isfinite(m['fusion_probability'])

normal=(fusion_ok & fusion_single &
        (m['fusion_seed_std'].to_numpy(float)<=cutoff('fusion_seed_std')) &
        (m['sharp_aia_probability_gap'].to_numpy(float)<=cutoff('sharp_aia_probability_gap')) &
        (m['fusion_entropy'].to_numpy(float)<=cutoff('fusion_entropy')))
degraded=(~normal)&sharp_ok
abstain=~(normal|degraded)

m['v3_state']=np.where(normal,'NORMAL',np.where(degraded,'DEGRADED','ABSTAIN'))
m['v3_prediction']=np.where(normal,m['sharp_aia_alarm'],np.where(degraded,m['sharp_alarm'],np.nan))

# ------------------------------------------------------------
# 1) Verify AR-disjointness against training
# ------------------------------------------------------------
train=cases[cases.role.astype(str).eq('train')].copy()
train_regions=set(train.region_component_id.dropna().astype(str))
ar_overlap=[]
for role,g in cases.groupby('role',sort=False):
    regs=set(g.region_component_id.dropna().astype(str))
    overlap=train_regions & regs
    ar_overlap.append({'role':role,'cases':len(g),'region_components':len(regs),
                       'overlap_with_train_region_components':len(overlap)})
ar_overlap=pd.DataFrame(ar_overlap)
ar_overlap.to_csv(OUT/'region_component_overlap_audit.csv',index=False)

# ------------------------------------------------------------
# 2) AR-level sensitivity
# A positive AR = at least one positive forecast window in that regime.
# Detection = at least one alert in that AR across all windows.
# ------------------------------------------------------------
ar_rows=[]
for rg,g in m.groupby('regime',sort=False):
    for ar,h in g.groupby('region_component_id'):
        positive_ar=bool(h.label_final.eq(1).any())
        for system,col in [('sharp','sharp_alarm'),('aia','aia_alarm'),('fusion','sharp_aia_alarm')]:
            ar_rows.append({
                'regime':rg,'region_component_id':str(ar),'system':system,
                'cases':len(h),'positive_windows':int(h.label_final.sum()),
                'positive_ar':positive_ar,'any_alert':bool(h[col].eq(1).any()),
                'false_alert_ar':bool((~positive_ar) and h[col].eq(1).any())
            })
        issued=h.v3_prediction.notna()
        ar_rows.append({
            'regime':rg,'region_component_id':str(ar),'system':'fallback_v3_q90',
            'cases':len(h),'positive_windows':int(h.label_final.sum()),
            'positive_ar':positive_ar,'any_alert':bool((h.loc[issued,'v3_prediction']==1).any()),
            'false_alert_ar':bool((~positive_ar) and (h.loc[issued,'v3_prediction']==1).any())
        })
ar_cases=pd.DataFrame(ar_rows)
ar_cases.to_csv(OUT/'ar_level_cases.csv.gz',index=False,compression={'method':'gzip','mtime':0})

ar_summary=[]
for (rg,system),g in ar_cases.groupby(['regime','system'],sort=False):
    pos=g.positive_ar
    neg=~pos
    detected=(g.any_alert&pos)
    false_ar=(g.any_alert&neg)
    ar_summary.append({
        'regime':rg,'system':system,'region_components':len(g),
        'positive_region_components':int(pos.sum()),
        'negative_region_components':int(neg.sum()),
        'detected_positive_region_components':int(detected.sum()),
        'missed_positive_region_components':int((pos&~g.any_alert).sum()),
        'ar_detection_rate':float(detected.sum()/pos.sum()) if pos.sum() else np.nan,
        'false_alert_region_components':int(false_ar.sum()),
        'ar_false_alert_rate':float(false_ar.sum()/neg.sum()) if neg.sum() else np.nan,
    })
ar_summary=pd.DataFrame(ar_summary)
ar_summary.to_csv(OUT/'ar_level_summary.csv',index=False)

# ------------------------------------------------------------
# 3) Exact event-disjoint analysis if retained event linkage exists
# ------------------------------------------------------------
candidate_paths=[
    ROOT/'gray-box-solar-flare-forecasting/data/processed/gray_box_outcomes_v2/candidate_outcomes.csv.gz',
    ROOT/'data/processed/gray_box_outcomes_v2/candidate_outcomes.csv.gz',
]
event_paths=[
    ROOT/'gray-box-solar-flare-forecasting/data/processed/gray_box_outcomes_v2/adjudicated_events.csv.gz',
    ROOT/'data/processed/gray_box_outcomes_v2/adjudicated_events.csv.gz',
]
candidate_path=next((p for p in candidate_paths if p.is_file()),None)
event_path=next((p for p in event_paths if p.is_file()),None)

if candidate_path is None:
    hits=[p for p in ROOT.glob('**/gray_box_outcomes_v2/candidate_outcomes.csv.gz')]
    candidate_path=hits[0] if len(hits)==1 else None
if event_path is None:
    hits=[p for p in ROOT.glob('**/gray_box_outcomes_v2/adjudicated_events.csv.gz')]
    event_path=hits[0] if len(hits)==1 else None

event_status={
    'candidate_outcomes_path':str(candidate_path) if candidate_path else None,
    'adjudicated_events_path':str(event_path) if event_path else None,
    'exact_event_linkage_available':False,
    'lead_time_available':False,
}

if candidate_path is not None:
    co=pd.read_csv(candidate_path,low_memory=False)
    require({'forecast_case_id','horizon_hours','primary_event_ids'}.issubset(co.columns),
            'Candidate outcome file lacks exact event-link fields')
    co=co[co.horizon_hours.eq(72)][['forecast_case_id','primary_event_ids']].copy()
    co['forecast_case_id']=co['forecast_case_id'].astype(str)
    joined=m.merge(co,on='forecast_case_id',how='left',validate='one_to_one')
    event_status['exact_event_linkage_available']=True

    event_rows=[]
    for row in joined.itertuples():
        ids=[x for x in str(row.primary_event_ids).split(';') if x and x!='nan']
        for eid in ids:
            event_rows.append({
                'event_id':eid,'forecast_case_id':row.forecast_case_id,'regime':row.regime,
                'region_component_id':str(row.region_component_id),'issue_dt':row.issue_dt,
                'sharp_alert':row.sharp_alarm,'aia_alert':row.aia_alarm,'fusion_alert':row.sharp_aia_alarm,
                'v3_prediction':row.v3_prediction,'v3_state':row.v3_state,
            })
    links=pd.DataFrame(event_rows)
    links.to_csv(OUT/'event_window_links.csv.gz',index=False,compression={'method':'gzip','mtime':0})

    if len(links):
        if event_path is not None:
            ev=pd.read_csv(event_path,low_memory=False)
            require('event_id' in ev.columns,'Adjudicated events lacks event_id')
            start_col='start_time' if 'start_time' in ev.columns else ('time' if 'time' in ev.columns else None)
            if start_col:
                ev['event_start_utc']=pd.to_datetime(ev[start_col],utc=True,errors='coerce')
                links=links.merge(ev[['event_id','event_start_utc']],on='event_id',how='left',validate='many_to_one')
                event_status['lead_time_available']=bool(links.event_start_utc.notna().any())
                links['lead_hours']=(links.event_start_utc-pd.to_datetime(links.issue_dt,utc=True)).dt.total_seconds()/3600

        summaries=[]
        for rg,g in links.groupby('regime',sort=False):
            for system,col in [('sharp','sharp_alert'),('aia','aia_alert'),('fusion','fusion_alert'),('fallback_v3_q90','v3_prediction')]:
                per=[]
                for eid,h in g.groupby('event_id'):
                    alerts=pd.to_numeric(h[col],errors='coerce')
                    detected=bool((alerts==1).any())
                    lead=np.nan
                    if detected and 'lead_hours' in h.columns:
                        z=h.loc[alerts.eq(1),'lead_hours'].dropna()
                        if len(z): lead=float(z.max())
                    per.append((eid,detected,lead))
                pp=pd.DataFrame(per,columns=['event_id','detected','lead_hours'])
                summaries.append({
                    'regime':rg,'system':system,'unique_events':len(pp),
                    'detected_events':int(pp.detected.sum()),
                    'missed_events':int((~pp.detected).sum()),
                    'event_detection_rate':float(pp.detected.mean()) if len(pp) else np.nan,
                    'median_max_lead_hours_detected':float(pp.loc[pp.detected,'lead_hours'].median()) if pp.loc[pp.detected,'lead_hours'].notna().any() else np.nan,
                })
        pd.DataFrame(summaries).to_csv(OUT/'event_level_summary.csv',index=False)
else:
    pd.DataFrame(columns=['regime','system','unique_events','detected_events','missed_events',
                          'event_detection_rate','median_max_lead_hours_detected']).to_csv(
        OUT/'event_level_summary.csv',index=False)

(OUT/'event_artifact_status.json').write_text(json.dumps(event_status,indent=2)+'\n')

protocol={
    'status':'POSTHOC_AR_EVENT_DISJOINT_AUDIT_COMPLETE',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'horizon_hours':72,
    'new_model_fitting':False,'new_threshold_tuning':False,'new_policy_selection':False,'new_policy_redesign':False,
    'ar_definition':'region_component_id from frozen multimodal case manifest',
    'ar_disjoint_test':'verify zero mapped region_component_id overlap between train and each later role; then report AR-level detection/false-alert sensitivity',
    'event_definition':'exact primary_event_ids from adjudicated 72h candidate outcomes when retained artifact is available',
    'event_status':event_status,
    'evidence_boundary':{'cycle24':'development/descriptive','cycle25':'diagnostic/post-hoc for v3','2026':'already-spent locked evidence; no redesign'},
    'source_sha256':{'master_predictions':MASTER_SHA,'sharp_predictions':SHARP_SHA,'cases':sha256(CASES)},
    'outputs':['region_component_overlap_audit.csv','ar_level_cases.csv.gz','ar_level_summary.csv',
               'event_level_summary.csv','event_artifact_status.json'],
}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== AR / EVENT DISJOINT AUDIT COMPLETE =====')
print('\nRegion overlap with train:')
print(ar_overlap.to_string(index=False))
print('\nAR-level summary:')
print(ar_summary.round(6).to_string(index=False))
print('\nEvent artifact status:')
print(json.dumps(event_status,indent=2))
if (OUT/'event_level_summary.csv').is_file():
    evs=pd.read_csv(OUT/'event_level_summary.csv')
    if len(evs):
        print('\nEvent-level summary:')
        print(evs.round(6).to_string(index=False))
print('\nOutputs:',OUT)


## Interpretation guardrails

- Zero overlap in `region_component_id` supports a mapped-AR-disjoint sensitivity, not proof of perfect physical independence between all solar structures.
- AR-level detection treats an active region as detected if any window in that region alerts; this avoids counting every overlapping positive window as an independent event.
- Exact event-level results are produced only if retained `primary_event_ids` are available. The notebook will not fabricate event identities from overlapping windows.
- Cycle 25 v3 results are post-hoc because Cycle 25 informed the v3 redesign. 2026 must not be used to retune the policy.
